In [24]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [8]:
# Загружаем исходный CSV
df = pd.read_excel('bank.xlsx')

print(f"Загружено строк: {len(df)}")
print(f"Колонки: {df.columns.tolist()}")
df.head()

Загружено строк: 116201
Колонки: ['Account No', 'DATE', 'TRANSACTION DETAILS', 'CHQ.NO.', 'VALUE DATE', 'WITHDRAWAL AMT', 'DEPOSIT AMT', 'BALANCE AMT', '.']


,Account No,DATE,TRANSACTION DETAILS,CHQ.NO.,VALUE DATE,WITHDRAWAL AMT,DEPOSIT AMT,BALANCE AMT,.
0,409000611074',2017-06-29,TRF FROM Indiaforensic SERVICES,NaN,2017-06-29,NaN,1000000.0,1000000.0,.
1,409000611074',2017-07-05,TRF FROM Indiaforensic SERVICES,NaN,2017-07-05,NaN,1000000.0,2000000.0,.
2,409000611074',2017-07-18,FDRL/INTERNAL FUND TRANSFE,NaN,2017-07-18,NaN,500000.0,2500000.0,.
3,409000611074',2017-08-01,TRF FRM Indiaforensic SERVICES,NaN,2017-08-01,NaN,3000000.0,5500000.0,.
4,409000611074',2017-08-16,FDRL/INTERNAL FUND TRANSFE,NaN,2017-08-16,NaN,500000.0,6000000.0,.


In [15]:
# Преобразуем DATE в datetime
df['DATE'] = pd.to_datetime(df['DATE'])
df['VALUE DATE'] = pd.to_datetime(df['VALUE DATE'])

print(f"Диапазон дат: {df['DATE'].min()} до {df['DATE'].max()}")
print(f"Уникальных дат: {df['DATE'].nunique()}")

# Проверяем пропуски в датах
print(f"Пропусков в DATE: {df['DATE'].isna().sum()}")

Диапазон дат: 2015-01-01 00:00:00 до 2019-03-05 00:00:00
Уникальных дат: 1294
Пропусков в DATE: 0


In [14]:
duplicates_before = df.duplicated().sum()
print(f"Дубликатов строк до удаления: {duplicates_before}")

df = df.drop_duplicates()

if 'transaction_id' in df.columns:
    df = df.drop_duplicates(subset=['transaction_id'])
    print("Удалены дубликаты по transaction_id")

print(f"После удаления дубликатов: {len(df)} строк")

Дубликатов строк до удаления: 39
После удаления дубликатов: 116162 строк


In [16]:
# Заполняем NaN нулями в суммах
df['WITHDRAWAL AMT'] = df['WITHDRAWAL AMT'].fillna(0)
df['DEPOSIT AMT'] = df['DEPOSIT AMT'].fillna(0)

# Создаём колонку с суммой транзакции (со знаком)
# Положительные = приток (DEPOSIT), отрицательные = отток (WITHDRAWAL)
df['AMOUNT'] = df['DEPOSIT AMT'] - df['WITHDRAWAL AMT']

# Колонка тип транзакции (приток/отток)
df['IS_INFLOW'] = (df['AMOUNT'] > 0).astype(int)

# Отдельные колонки для притока и оттока (как в плане)
df['INFLOW_AMOUNT'] = df['DEPOSIT AMT']
df['OUTFLOW_AMOUNT'] = df['WITHDRAWAL AMT']

# Проверяем
print(f"Притоков (INFLOW): {df['IS_INFLOW'].sum()}")
print(f"Оттоков (OUTFLOW): {(df['IS_INFLOW'] == 0).sum()}")
print(f"\nПример транзакций:")
df[['DATE', 'TRANSACTION DETAILS', 'WITHDRAWAL AMT', 'DEPOSIT AMT', 'AMOUNT', 'IS_INFLOW']].head(10)

Притоков (INFLOW): 62637
Оттоков (OUTFLOW): 53525

Пример транзакций:


,DATE,TRANSACTION DETAILS,WITHDRAWAL AMT,DEPOSIT AMT,AMOUNT,IS_INFLOW
0,2017-06-29,TRF FROM Indiaforensic SERVICES,0.0,1000000.0,1000000.0,1
1,2017-07-05,TRF FROM Indiaforensic SERVICES,0.0,1000000.0,1000000.0,1
2,2017-07-18,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,500000.0,1
3,2017-08-01,TRF FRM Indiaforensic SERVICES,0.0,3000000.0,3000000.0,1
4,2017-08-16,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,500000.0,1
5,2017-08-16,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,500000.0,1
6,2017-08-16,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,500000.0,1
7,2017-08-16,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,500000.0,1
8,2017-08-16,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,500000.0,1
9,2017-08-16,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,500000.0,1


In [17]:
# Отсекаем экстремальные выбросы по абсолютной сумме
amounts = df['AMOUNT'].abs()

# Метод 99.9 перцентиля
upper_limit = amounts.quantile(0.999)
print(f"99.9% перцентиль суммы: {upper_limit:,.2f}")

# Удаляем выбросы
before = len(df)
df = df[amounts <= upper_limit]
print(f"Удалено выбросов: {before - len(df)}")

# Дополнительно: убираем нулевые транзакции (если есть)
zero_transactions = (df['WITHDRAWAL AMT'] == 0) & (df['DEPOSIT AMT'] == 0)
if zero_transactions.any():
    print(f"Нулевых транзакций: {zero_transactions.sum()}")
    df = df[~zero_transactions]
    print(f"После удаления нулевых: {len(df)} строк")

99.9% перцентиль суммы: 99,416,203.85
Удалено выбросов: 117


In [21]:
# Очищаем номер счета
df['account_no'] = df['Account No'].astype(str).str.replace("'", "")

# Создаём ID для таблиц (для SQLite схемы)
df['transaction_id'] = range(1, len(df) + 1)
df['client_id'] = 1  # пока один клиент
df['account_id'] = df['account_no'].astype('category').cat.codes + 1

# Переименовываем колонки под единый формат
df = df.rename(columns={
    'DATE': 'transaction_date',
    'TRANSACTION DETAILS': 'description',
    'VALUE DATE': 'value_date',
    'BALANCE AMT': 'balance_amount'
})

# Убеждаемся, что все нужные колонки существуют
print("Проверка наличия колонок:")
print(f"- WITHDRAWAL AMT: {'WITHDRAWAL AMT' in df.columns}")
print(f"- DEPOSIT AMT: {'DEPOSIT AMT' in df.columns}")
print(f"- is_inflow: {'is_inflow' in df.columns}")

# Создаём колонки с правильными именами (если их нет)
df['withdrawal_amount'] = df['WITHDRAWAL AMT']
df['deposit_amount'] = df['DEPOSIT AMT']

# Если is_inflow отсутствует, создаём заново
if 'is_inflow' not in df.columns:
    print("Создаю is_inflow заново...")
    df['is_inflow'] = (df['deposit_amount'] > 0).astype(int)
    df['inflow_amount'] = df['deposit_amount']
    df['outflow_amount'] = df['withdrawal_amount']
    df['amount'] = df['deposit_amount'] - df['withdrawal_amount']

# Список колонок для сохранения
columns_for_clean = [
    'transaction_id', 'account_no', 'client_id', 'account_id',
    'transaction_date', 'value_date', 'description',
    'withdrawal_amount', 'deposit_amount', 'balance_amount',
    'amount', 'is_inflow', 'inflow_amount', 'outflow_amount'
]

# Проверяем какие колонки есть
existing_cols = [col for col in columns_for_clean if col in df.columns]
missing_cols = [col for col in columns_for_clean if col not in df.columns]

print(f"\nСуществующие колонки: {existing_cols[:5]}...")
print(f"Отсутствуют: {missing_cols}")

if missing_cols:
    print("\n⚠️ Внимание! Отсутствуют колонки, создаю их:")
    for col in missing_cols:
        if col == 'balance_amount':
            df['balance_amount'] = 0
        elif col == 'value_date':
            df['value_date'] = df['transaction_date']
        elif col in ['inflow_amount', 'outflow_amount', 'amount', 'is_inflow']:
            # Уже создали выше
            pass
        else:
            df[col] = 0

# Снова проверяем существующие колонки
existing_cols = [col for col in columns_for_clean if col in df.columns]
df_clean = df[existing_cols]

print(f"\n✅ Итоговое количество строк: {len(df_clean)}")
print(f"✅ Колонки: {df_clean.columns.tolist()}")
df_clean.head()

Проверка наличия колонок:
- WITHDRAWAL AMT: True
- DEPOSIT AMT: True
- is_inflow: False
Создаю is_inflow заново...

Существующие колонки: ['transaction_id', 'account_no', 'client_id', 'account_id', 'transaction_date']...
Отсутствуют: []

✅ Итоговое количество строк: 116045
✅ Колонки: ['transaction_id', 'account_no', 'client_id', 'account_id', 'transaction_date', 'value_date', 'description', 'withdrawal_amount', 'deposit_amount', 'balance_amount', 'amount', 'is_inflow', 'inflow_amount', 'outflow_amount']


,transaction_id,account_no,client_id,account_id,transaction_date,value_date,description,withdrawal_amount,deposit_amount,balance_amount,amount,is_inflow,inflow_amount,outflow_amount
0,1,409000611074,1,10,2017-06-29,2017-06-29,TRF FROM Indiaforensic SERVICES,0.0,1000000.0,1000000.0,1000000.0,1,1000000.0,0.0
1,2,409000611074,1,10,2017-07-05,2017-07-05,TRF FROM Indiaforensic SERVICES,0.0,1000000.0,2000000.0,1000000.0,1,1000000.0,0.0
2,3,409000611074,1,10,2017-07-18,2017-07-18,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,2500000.0,500000.0,1,500000.0,0.0
3,4,409000611074,1,10,2017-08-01,2017-08-01,TRF FRM Indiaforensic SERVICES,0.0,3000000.0,5500000.0,3000000.0,1,3000000.0,0.0
4,5,409000611074,1,10,2017-08-16,2017-08-16,FDRL/INTERNAL FUND TRANSFE,0.0,500000.0,6000000.0,500000.0,1,500000.0,0.0


In [26]:
# Сохраняем как transactions_clean.csv
output_path = 'transactions_clean.csv'
df_clean.to_csv(output_path, index=False)

# Сразу проверяем, что сохранилось правильно
check_df = pd.read_csv(output_path)
print(f"✅ Сохранено: {output_path}")
print(f"Размер файла: {os.path.getsize(output_path) / 1024**2:.2f} MB")
print(f"\nСтатистика:")
print(f"  - Транзакций: {len(check_df):,}")
print(f"  - Колонки: {check_df.columns.tolist()}")
print(f"  - Притоки (is_inflow=1): {check_df['is_inflow'].sum():,}")
print(f"  - Оттоки (is_inflow=0): {(check_df['is_inflow'] == 0).sum():,}")

# Показываем первые строки
print("\nПервые 3 строки:")
print(check_df[['transaction_id', 'is_inflow', 'amount', 'inflow_amount', 'outflow_amount']].head(3))

✅ Сохранено: transactions_clean.csv
Размер файла: 13.81 MB

Статистика:
  - Транзакций: 116,045
  - Колонки: ['transaction_id', 'account_no', 'client_id', 'account_id', 'transaction_date', 'value_date', 'description', 'withdrawal_amount', 'deposit_amount', 'balance_amount', 'amount', 'is_inflow', 'inflow_amount', 'outflow_amount']
  - Притоки (is_inflow=1): 62,592
  - Оттоки (is_inflow=0): 53,453

Первые 3 строки:
   transaction_id  is_inflow     amount  inflow_amount  outflow_amount
0               1          1  1000000.0      1000000.0             0.0
1               2          1  1000000.0      1000000.0             0.0
2               3          1   500000.0       500000.0             0.0
